# Intro 01 — Foundations: Sets, Events, and Notation

Practice notebook for the **"Foundations: Sets, Events, and Notation"** section of the stats course PDF.

## What you will learn

This notebook recaps the theory from the PDF section, then **verifies it with Python**.
Each part ends with a **"Your turn"** prompt; the full **Problem Set** at the end has
**click-to-reveal solutions**.

## How to use

1. **Read** each markdown cell, then **run** the code beneath it (`Shift+Enter`).
2. **Change parameters** and re-run — statistics is about *relationships*, not memorized formulas.
3. LaTeX uses \( \) for inline math and \[ \] / $$ $$ for display math (KaTeX-friendly).

Let's begin.


---
## Setup — run this first

NumPy for numerics, SciPy for exact distributions, Matplotlib for plots.


In [ ]:
# If anything is missing, uncomment and run once:
# !pip install numpy scipy matplotlib

import numpy as np
import matplotlib.pyplot as plt
import scipy
from scipy import stats

np.random.seed(42)
rng = np.random.default_rng(42)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

print("Ready. NumPy", np.__version__, "| SciPy", scipy.__version__)


---
## Part 1 — Sets and Events

Probability is built on **sets** of outcomes.

- The **sample space** \(\Omega\) is the set of all possible outcomes of a random experiment.
- An **event** is a subset \(A \subseteq \Omega\) of outcomes for which something of interest happens.

**Example (one coin toss).**
$$\Omega = \{\,\text{H},\,\text{T}\,\},\qquad A = \{\text{H}\}\subseteq\Omega.$$
A probability model will later assign numbers like \(P(A)=0.5\), meaning we expect heads half the time.

Below we simulate many coin tosses and check that the *relative frequency* of the event \(A=\{\text{H}\}\) approaches \(0.5\) as the number of tosses grows (a preview of the frequentist interpretation).


In [ ]:
# Simulate a fair-coin experiment and watch the relative frequency of the event A={H}
rng = np.random.default_rng(42)
Omega = np.array(["H", "T"])           # the sample space
A = np.array(["H"])                    # the event "heads"

n_tosses = 50_000
draws = rng.choice(Omega, size=n_tosses, p=[0.5, 0.5])

# Relative frequency of A: |A observed| / n
counts = np.cumsum(draws[:, None] == A)[:, 0]
rel_freq = counts / np.arange(1, n_tosses + 1)

print("Sample space Omega =", Omega.tolist())
print("Event A =", A.tolist())
print(f"After {n_tosses:,} tosses, P(A) ≈ {rel_freq[-1]:.4f}  (theory = 0.5)")

# Plot convergence
fig, ax = plt.subplots()
ax.plot(rel_freq, lw=1.0, label="relative frequency of A")
ax.axhline(0.5, color="tab:red", ls="--", label="theory 0.5")
ax.set_xlabel("number of tosses")
ax.set_ylabel("relative frequency")
ax.set_title(r"Frequentist estimate of $P(A)$ for $A=\{H\}$")
ax.legend()
ax.set_ylim(0.3, 0.7)
plt.show()


---
## Part 2 — Set Operations

Given events \(A,B\subseteq\Omega\) we use the standard set operations:

- **Union:** \(A\cup B\) — "$A$ occurs OR $B$ occurs (or both)".
- **Intersection:** \(A\cap B\) — "$A$ AND $B$ occur simultaneously".
- **Complement:** \(A^c=\Omega\setminus A\) — "$A$ does not occur".

**Example (die roll).** With
$$\Omega=\{1,2,3,4,5,6\},\quad A=\{2,4,6\},\quad B=\{4,5,6\},$$
the theory gives
$$A\cup B=\{2,4,5,6\},\quad A\cap B=\{4,6\},\quad A^c=\{1,3,5\}.$$

The code below constructs these sets explicitly in Python and also verifies two
**inclusion–exclusion** identities by counting elements:
$$|A\cup B| = |A| + |B| - |A\cap B|,\qquad |A^c| = |\Omega| - |A|.$$
We then run a Monte-Carlo estimate of \(P(A\cup B)\) on a fair die and compare it to
\(|A\cup B|/|\Omega| = 4/6\).


In [ ]:
# Exact set operations on a die-roll sample space
Omega = np.array([1, 2, 3, 4, 5, 6])
A = np.array([2, 4, 6])           # even outcomes
B = np.array([4, 5, 6])           # outcomes > 3

def intersect(x, y):
    return np.intersect1d(x, y)

def union(x, y):
    return np.union1d(x, y)

def complement(x, omega):
    return np.setdiff1d(omega, x)

AuB = union(A, B)
AiB = intersect(A, B)
Ac = complement(A, Omega)

print("Omega      =", Omega.tolist())
print("A (even)   =", A.tolist())
print("B (>3)     =", B.tolist())
print("A u B      =", AuB.tolist(), " (theory {2,4,5,6})")
print("A n B      =", AiB.tolist(), " (theory {4,6})")
print("A^c        =", Ac.tolist(),  " (theory {1,3,5})")

# Inclusion–exclusion by counting
assert len(AuB) == len(A) + len(B) - len(AiB)
assert len(Ac) == len(Omega) - len(A)
print("\nInclusion–exclusion: |A u B| = |A| + |B| - |A n B| =",
      len(A), "+", len(B), "-", len(AiB), "=", len(AuB), "✓")

# Monte-Carlo check: P(A u B) should approach |A u B| / |Omega| = 4/6
rng = np.random.default_rng(7)
n = 200_000
rolls = rng.choice(Omega, size=n)
p_AuB_sim = np.mean(np.isin(rolls, AuB))
p_AuB_theory = len(AuB) / len(Omega)
print(f"\nP(A u B):  simulated = {p_AuB_sim:.4f},  theory = {p_AuB_theory:.4f}  (4/6)")


---
## Part 3 — De Morgan's Laws (numerical verification)

The set operations satisfy **De Morgan's laws**, which connect complements with
unions/intersections:
$$(A\cup B)^c = A^c\cap B^c,\qquad (A\cap B)^c = A^c\cup B^c.$$

Verbally: *the complement of a union is the intersection of the complements*, and vice versa.
These laws are used constantly to rewrite events.

We verify both identities numerically by comparing the underlying outcome sets, and we
also confirm them probabilistically: simulating die rolls and checking that the relative
frequencies of both sides agree.


In [ ]:
# Verify De Morgan's laws on the die-roll space
Omega = np.array([1, 2, 3, 4, 5, 6])
A = np.array([2, 4, 6])
B = np.array([4, 5, 6])

LHS1 = complement(union(A, B), Omega)        # (A u B)^c
RHS1 = intersect(complement(A, Omega), complement(B, Omega))   # A^c n B^c
LHS2 = complement(intersect(A, B), Omega)    # (A n B)^c
RHS2 = union(complement(A, Omega), complement(B, Omega))       # A^c u B^c

print("(A u B)^c =", LHS1.tolist(), "| A^c n B^c =", RHS1.tolist(),
      "=> equal:", np.array_equal(np.sort(LHS1), np.sort(RHS1)))
print("(A n B)^c =", LHS2.tolist(), "| A^c u B^c =", RHS2.tolist(),
      "=> equal:", np.array_equal(np.sort(LHS2), np.sort(RHS2)))

# Probabilistic check on simulated die rolls
rng = np.random.default_rng(123)
n = 100_000
rolls = rng.choice(Omega, size=n)
in_A = np.isin(rolls, A)
in_B = np.isin(rolls, B)

p_LHS1 = np.mean(~(in_A | in_B))          # (A u B)^c
p_RHS1 = np.mean((~in_A) & (~in_B))       # A^c n B^c
p_LHS2 = np.mean(~(in_A & in_B))          # (A n B)^c
p_RHS2 = np.mean((~in_A) | (~in_B))       # A^c u B^c

print("\nDe Morgan 1: P((A u B)^c) = %.4f  vs  P(A^c n B^c) = %.4f" % (p_LHS1, p_RHS1))
print("De Morgan 2: P((A n B)^c) = %.4f  vs  P(A^c u B^c) = %.4f" % (p_LHS2, p_RHS2))


---
## Part 4 — Sigma-Algebras (optional intuition)

In rigorous probability we don't declare *every* imaginable subset of \(\Omega\) to be an
event — only those in a distinguished collection \(\mathcal{F}\), called a
**sigma-algebra**. A sigma-algebra must be

1. closed under **complements**: if \(A\in\mathcal{F}\) then \(A^c\in\mathcal{F}\);
2. closed under **countable unions**: if \(A_1,A_2,\dots\in\mathcal{F}\) then
   \(\bigcup_i A_i\in\mathcal{F}\);
3. it contains \(\Omega\) itself (equivalently \(\emptyset\)).

For finite or countable \(\Omega\) we usually take \(\mathcal{F}=2^{\Omega}\) (all
subsets), which is automatically a sigma-algebra. The smallest non-trivial example is the
**trivial sigma-algebra** \(\{\emptyset,\Omega\}\); the next simplest is generated by a
single event \(A\):
$$\sigma(A) = \{\,\emptyset,\, A,\, A^c,\, \Omega\,\}.$$

The code below enumerates \(\sigma(A)\) for a coin toss and checks closure under complements
and pairwise unions.


In [ ]:
# Enumerate sigma(A) for Omega = {H, T}, A = {H}
Omega = np.array(["H", "T"])
A = np.array(["H"])

def sigma_generated_by(A, Omega):
    # sigma(A) = {emptyset, A, A^c, Omega}
    return [
        np.array([], dtype=Omega.dtype),     # emptyset
        np.sort(A),                          # A
        np.sort(complement(A, Omega)),        # A^c
        np.sort(Omega),                       # Omega
    ]

F = sigma_generated_by(A, Omega)
print("sigma(A) has", len(F), "elements:")
for i, S in enumerate(F, 1):
    print(f"  {i}. {set(S.tolist()) if S.size else '{}'}")

# Closure checks
def closed_under_complements(F, Omega):
    return all(any(np.array_equal(np.sort(complement(S, Omega)), np.sort(T)) for T in F)
               for S in F)

def closed_under_pairwise_unions(F):
    return all(any(np.array_equal(np.sort(union(S, T)), np.sort(U)) for U in F)
               for S in F for T in F)

print("\nClosed under complements?   ", closed_under_complements(F, Omega))
print("Closed under pairwise unions?", closed_under_pairwise_unions(F))

# Compare with the full power set 2^Omega (4 subsets for |Omega|=2)
from itertools import combinations
power_set = [np.array(c, dtype=Omega.dtype) for k in range(len(Omega)+1)
             for c in combinations(Omega.tolist(), k)]
print("\nFull power set 2^Omega has", len(power_set), "subsets (also a sigma-algebra).")


---
## Your turn

Before opening the Problem Set below:

- Pick a new sample space of your own (e.g. two dice, a 52-card deck, the days of a week)
  and define two events \(A\) and \(B\).
- Compute \(A\cup B\), \(A\cap B\), \(A^c\) by hand, then verify with `np.union1d` /
  `np.intersect1d` / `np.setdiff1d`.
- Confirm both of **De Morgan's laws** numerically on your example.
- Write down \(\sigma(A)\) and check it is closed under complements and unions.


---
# Problem Set

**Try each problem before revealing the solution.**


## Problems

1. Let \(\Omega=\{1,2,3,4,5,6\}\) be the outcome of one fair die. Define \(A=\{1,2,3\}\) and \(B=\{3,4\}\). List the outcomes in \(A\cup B\), \(A\cap B\), \(A\cup B)^c\), and \(A\cap B)^c\).

2. For the same \(\Omega\), \(A\), \(B\) as Problem 1, verify De Morgan's law \((A\cup B)^c = A^c\cap B^c\) by listing both sides explicitly.

3. Toss a fair coin three times, so \(\Omega=\{\text{HHH},\text{HHT},\dots,\text{TTT}\}\) (8 outcomes). Let \(A=\)"at least two heads" and \(B=\)"the first toss is H". Compute \(|A|\), \(|B|\), \(|A\cap B|\), and \(|A\cup B|\), then check inclusion–exclusion.

4. Give an example of a sigma-algebra on \(\Omega=\{a,b,c\}\) that has exactly 4 elements, and confirm it is closed under complements and pairwise unions.

5. Simulate 100,000 rolls of a fair six-sided die and estimate \(P(A\cup B)\) for \(A=\{2,4,6\}\), \(B=\{4,5,6\}\). Compare with the exact value.


<details>
<summary><strong>Reveal solutions</strong></summary>

**1.** \(A\cup B=\{1,2,3,4\}\). \(A\cap B=\{3\}\). \((A\cup B)^c=\{5,6\}\). \((A\cap B)^c=\{1,2,4,5,6\}\).

**2.** \(A^c=\{4,5,6\}\), \(B^c=\{1,2,5,6\}\), so \(A^c\cap B^c=\{5,6\}\), which equals \((A\cup B)^c=\{5,6\}\) from Problem 1. ✓

**3.** \(A=\{\text{HHH},\text{HHT},\text{HTH},\text{THH}\}\) so \(|A|=4\). \(B=\{\text{HHH},\text{HHT},\text{HTH},\text{HTT}\}\) so \(|B|=4\). \(A\cap B=\{\text{HHH},\text{HHT},\text{HTH}\}\), \(|A\cap B|=3\). Inclusion–exclusion: \(|A\cup B|=4+4-3=5\). The union is \(\{\text{HHH},\text{HHT},\text{HTH},\text{THH},\text{HTT}\}\), which indeed has 5 elements.

**4.** Take any partition \(\Omega=E_1\sqcup E_2\) with \(E_1=\{a\}\), \(E_2=\{b,c\}\). The sigma-algebra generated by \(E_1\) is \(\mathcal{F}=\{\emptyset,\,\{a\},\,\{b,c\},\,\{a,b,c\}\}\), which has 4 elements. Complements: \(\{a\}^c=\{b,c\}\) and vice versa, \(\emptyset^c=\Omega\). Pairwise unions stay inside \(\mathcal{F}\) (e.g. \(\{a\}\cup\{b,c\}=\Omega\)). Closed ✓.

**5.** \(|A\cup B|=4\) so \(P(A\cup B)=4/6=2/3\approx 0.6667\). A simulation with `rng = np.random.default_rng(seed)` gives an estimate very close to \(0.6667\); the variance of the estimate is \(p(1-p)/n\approx 0.000222\), so the standard error is about \(0.015\).

</details>
